<a href="https://colab.research.google.com/github/anokhina-rgb/Pyfiles/blob/main/mp3splitter%2Bhandout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- 1. Install dependencies ---
!pip install openai-whisper transformers sentencepiece docx2txt python-docx fpdf pydub matplotlib nltk gTTS

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import whisper
from pydub import AudioSegment
from gtts import gTTS
import matplotlib.pyplot as plt
import numpy as np
from docx import Document
from transformers import MarianMTModel, MarianTokenizer
import zipfile, os
from google.colab import files

# --- 2. Upload MP3 ---
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"Uploaded file: {audio_file}")

# --- 3. Transcription with Whisper ---
model = whisper.load_model("small")  # or "base" for faster
result = model.transcribe(audio_file)
full_text = result["text"]
print("Transcribed text:\n", full_text[:500])

# --- 4. Sentence segmentation ---
from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(full_text)
print(f"Found {len(sentences)} sentences.")

# --- 5. Generate audio with 10s pauses ---
pause = AudioSegment.silent(duration=10000)  # 10 sec pause
audio_with_pauses = AudioSegment.silent(duration=0)
sentence_timestamps = []
current_time = 0

for i, sent in enumerate(sentences, 1):
    tts = gTTS(sent)
    tts.save(f"sent_{i}.mp3")
    seg = AudioSegment.from_mp3(f"sent_{i}.mp3")
    audio_with_pauses += seg + pause
    sentence_timestamps.append((current_time, current_time + len(seg), sent))
    current_time += len(seg) + len(pause)

audio_with_pauses.export("output_with_pauses.wav", format="wav")

# --- 6. Waveform full ---
plt.figure(figsize=(14, 5))
plt.plot(audio_with_pauses.get_array_of_samples(), alpha=0.7)
plt.title("Audio with pauses (full signal)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.savefig("wave_full.png")
plt.show()

# --- 7. Waveform with sentence markers ---
plt.figure(figsize=(14, 6))
samples = np.array(audio_with_pauses.get_array_of_samples())
plt.plot(samples, alpha=0.7)
for i, (start, end, _) in enumerate(sentence_timestamps, 1):
    x = int(start * audio_with_pauses.frame_rate / 1000)
    plt.axvline(x, color='red', linestyle='--', alpha=0.7)
    plt.text(x, samples.max() * 0.9, str(i), color='blue',
             fontsize=10, rotation=90, ha='center', va='bottom')

plt.title("Audio with sentence segmentation (red lines + numbers)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.savefig("wave_sentences.png")
plt.show()

# --- 8. Translation EN -> UKR (fallback to English if fail) ---
translated_sentences = []
try:
    model_name = 'Helsinki-NLP/opus-mt-en-uk'
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model_mt = MarianMTModel.from_pretrained(model_name)
    for sent in sentences:
        batch = tokenizer([sent], return_tensors="pt", padding=True)
        translated = model_mt.generate(**batch)
        ukr_sent = tokenizer.decode(translated[0], skip_special_tokens=True)
        translated_sentences.append(ukr_sent)
except Exception as e:
    print("⚠️ Translation failed:", e)
    translated_sentences = sentences  # fallback

# --- 9. Save Word handout ---
doc = Document()
doc.add_heading("Handout", 0)

# Page 1
doc.add_heading("Page 1: Full text", level=1)
doc.add_paragraph(full_text)

# Page 2
doc.add_page_break()
doc.add_heading("Page 2: Numbered sentences with timestamps", level=1)
for i, (start, end, sent) in enumerate(sentence_timestamps, 1):
    doc.add_paragraph(f"{i}. [{start/1000:.1f}s - {end/1000:.1f}s] {sent}")

# Page 3
doc.add_page_break()
doc.add_heading("Page 3: Translation (EN->UKR)", level=1)
for i, sent in enumerate(translated_sentences, 1):
    doc.add_paragraph(f"{i}. {sent}")

doc.save("Handout.docx")

# --- 10. Save TXT with timestamps ---
with open("sentence_timestamps.txt", "w") as f:
    for i, (start, end, sent) in enumerate(sentence_timestamps, 1):
        f.write(f"{i}. [{start/1000:.1f}s - {end/1000:.1f}s] {sent}\n")

# --- 11. ZIP all results ---
zip_filename = "Handout_Files.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    for fname in ["output_with_pauses.wav", "wave_full.png", "wave_sentences.png",
                  "Handout.docx", "sentence_timestamps.txt"]:
        zipf.write(fname)

files.download(zip_filename)
